<a href="https://www.kaggle.com/code/skanderadamafi/abstract-generator-using-transformer?scriptVersionId=294843235" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# **1. Load and Explore the Dataset**

In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Layer, Dense, Embedding, Dropout
from tensorflow.keras.models import Model

2026-01-29 22:18:51.187284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769725131.407488      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769725131.469252      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769725131.993182      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769725131.993240      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769725131.993243      23 computation_placer.cc:177] computation placer alr

In [2]:
# Load the dataset
dataset = pd.read_csv('/kaggle/input/CORD-19-research-challenge/metadata.csv', low_memory=False)

# Take a random sample of 100 rows from the dataset
sampled_data = dataset.sample(n=100, random_state=42)

# Display the first few rows of the sampled data
sampled_data.head(10)

,cord_uid,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,mag_id,who_covidence_id,arxiv_id,pdf_json_files,pmc_json_files,url,s2_id
17948,ak20jg32,42a61efa32fae8fb14d1c8c1a8bc528c59e1583a,PMC,Development and Internal Validation of a Novel...,10.3389/fpsyt.2021.593710,PMC8172985,34093252,cc-by,Objective: The aim of our study was to identif...,2021-05-20,"Zhou, Jingjing; Zhou, Jia; Sun, Zuoli; Feng, L...",Front Psychiatry,NaN,NaN,NaN,document_parses/pdf_json/42a61efa32fae8fb14d1c...,document_parses/pmc_json/PMC8172985.xml.json,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8...,NaN
915932,d27cnei3,NaN,PMC; WHO,mRNA-1273: Acute disseminated encephalomyeliti...,10.1007/s40278-021-06200-0,PMC8617002,NaN,no-cc,NaN,2021-11-27,NaN,Reactions Weekly,NaN,NaN,NaN,NaN,document_parses/pmc_json/PMC8617002.xml.json,https://doi.org/10.1007/s40278-021-06200-0,244637117.0
456857,7fo988o7,NaN,WHO,The Joint Commission should reconsider its pos...,NaN,NaN,NaN,unk,NaN,2020,"Kroll, David S; Shah, Sejal B; Gorman, Janet M",Gen. hosp. psychiatr,NaN,#1074753,NaN,NaN,NaN,NaN,227251863.0
176672,vcz81w3o,NaN,Medline,Flattening the Curve of Prostate Cancer Progre...,10.1016/j.ijrobp.2020.04.028,NaN,32589984,unk,NaN,2020-07-15,"Mitin, Timur; Choudhury, Ananya","International journal of radiation oncology, b...",NaN,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.ijrobp.2020.04.028; ...,220118792.0
40038,3ewr26np,NaN,PMC,Vincristine: Various toxicities: case report,10.1007/s40278-016-19002-y,PMC7149258,NaN,no-cc,NaN,2016-06-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7...,NaN
922293,6unac0pg,36ada922d634878a962c1f555ad144969c168e08,BioRxiv; WHO,Single domain shark VNAR antibodies neutralize...,10.1101/2021.06.08.447530,NaN,NaN,biorxiv,Single domain shark VNAR antibodies can offer ...,2021-06-08,"Gauhar, Aziz; Privezentzev, Cyril V; Demydchuk...",bioRxiv,NaN,NaN,NaN,document_parses/pdf_json/36ada922d634878a962c1...,NaN,https://doi.org/10.1101/2021.06.08.447530,235396724.0
906718,15emtsrv,34914674ac6b8c2276dfa86b9ff649ab640d22f5,Medline; PMC,"Breastfeeding, Human Milk and COVID-19—What Do...",10.3389/fped.2020.613339,PMC7714759,33330294,cc-by,NaN,2020-11-20,"Mitoulas, Leon R.; Schärer-Hernández, Nania G....",Front Pediatr,NaN,NaN,NaN,document_parses/pdf_json/34914674ac6b8c2276dfa...,document_parses/pmc_json/PMC7714759.xml.json,https://www.ncbi.nlm.nih.gov/pubmed/33330294/;...,227060359.0
455876,fcc1cohr,NaN,WHO,Point-of-care ultrasound during the COVID-19 p...,NaN,NaN,NaN,unk,PURPOSE: The coronavirus disease-2019 (COVID-1...,2021,"Yuriditsky, Eugene; Saric, Muhamed; Horowitz, ...",Echocardiography,NaN,#1084702,NaN,NaN,NaN,NaN,231925832.0
775710,73xyjolv,86bb8af8dfc34b7a8e6ff25b0167a1bda02fa0e9,PMC; WHO,Joint Predictive Value of cTnI and NT-proBNP o...,10.2478/jtim-2021-0034,PMC8629414,NaN,cc-by-nc-nd,BACKGROUND AND OBJECTIVES: The pandemic of cor...,2021-09-28,"Weng, Haoyu; Yang, Fan; Zhang, Long; Jin, Han;...",J Transl Int Med,NaN,NaN,NaN,document_parses/pdf_json/86bb8af8dfc34b7a8e6ff...,document_parses/pmc_json/PMC8629414.xml.json,https://doi.org/10.2478/jtim-2021-0034,238220710.0
115080,k82ifpki,NaN,Medline,Glass hybrid (glass ionomer) versus composite ...,10.1016/j.jdent.2021.103689,NaN,33979577,unk,"OBJECTIVE This study compared survival, restor...",2021-05-09,"Schwendicke, Falk; Müller, Anne; Seifert, Tilm...",Journal of dentistry,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.1016/j.jdent.2021.103689; h...,234485613.0


In [3]:
sampled_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 17948 to 245130
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   cord_uid          100 non-null    object 
 1   sha               36 non-null     object 
 2   source_x          100 non-null    object 
 3   title             100 non-null    object 
 4   doi               67 non-null     object 
 5   pmcid             37 non-null     object 
 6   pubmed_id         51 non-null     object 
 7   license           100 non-null    object 
 8   abstract          72 non-null     object 
 9   publish_time      100 non-null    object 
 10  authors           97 non-null     object 
 11  journal           94 non-null     object 
 12  mag_id            0 non-null      float64
 13  who_covidence_id  38 non-null     object 
 14  arxiv_id          1 non-null      object 
 15  pdf_json_files    36 non-null     object 
 16  pmc_json_files    32 non-null     object 


# **2. Data Cleaning and Preprocessing**

In [4]:
# Remove irrelevant columns
columns_to_keep = ['cord_uid', 'title', 'abstract']
filtered_dataset = sampled_data[columns_to_keep]

# Display the remaining dataset
filtered_dataset.head(10)

,cord_uid,title,abstract
17948,ak20jg32,Development and Internal Validation of a Novel...,Objective: The aim of our study was to identif...
915932,d27cnei3,mRNA-1273: Acute disseminated encephalomyeliti...,NaN
456857,7fo988o7,The Joint Commission should reconsider its pos...,NaN
176672,vcz81w3o,Flattening the Curve of Prostate Cancer Progre...,NaN
40038,3ewr26np,Vincristine: Various toxicities: case report,NaN
922293,6unac0pg,Single domain shark VNAR antibodies neutralize...,Single domain shark VNAR antibodies can offer ...
906718,15emtsrv,"Breastfeeding, Human Milk and COVID-19—What Do...",NaN
455876,fcc1cohr,Point-of-care ultrasound during the COVID-19 p...,PURPOSE: The coronavirus disease-2019 (COVID-1...
775710,73xyjolv,Joint Predictive Value of cTnI and NT-proBNP o...,BACKGROUND AND OBJECTIVES: The pandemic of cor...
115080,k82ifpki,Glass hybrid (glass ionomer) versus composite ...,"OBJECTIVE This study compared survival, restor..."


In [5]:
# Print the percentage of missing values per column
print(filtered_dataset.isnull().sum() / len(filtered_dataset) * 100)

cord_uid     0.0
title        0.0
abstract    28.0
dtype: float64


In [6]:
# Drop rows with any missing data
cleaned_dataset = filtered_dataset.dropna()

# Display the shape of the cleaned dataset to verify
print(f"Original dataset shape: {filtered_dataset.shape}")
print(f"Cleaned dataset shape: {cleaned_dataset.shape}")

# Display a preview of the cleaned dataset
cleaned_dataset.head()

Original dataset shape: (100, 3)
Cleaned dataset shape: (72, 3)


,cord_uid,title,abstract
17948,ak20jg32,Development and Internal Validation of a Novel...,Objective: The aim of our study was to identif...
922293,6unac0pg,Single domain shark VNAR antibodies neutralize...,Single domain shark VNAR antibodies can offer ...
455876,fcc1cohr,Point-of-care ultrasound during the COVID-19 p...,PURPOSE: The coronavirus disease-2019 (COVID-1...
775710,73xyjolv,Joint Predictive Value of cTnI and NT-proBNP o...,BACKGROUND AND OBJECTIVES: The pandemic of cor...
115080,k82ifpki,Glass hybrid (glass ionomer) versus composite ...,"OBJECTIVE This study compared survival, restor..."


In [7]:
# Print the percentage of missing values per column
print(cleaned_dataset.isnull().sum() / len(cleaned_dataset) * 100)

cord_uid    0.0
title       0.0
abstract    0.0
dtype: float64


In [8]:
def clean_text(text):
    # Remove punctuation and special characters using regex
    text = re.sub(r'[^\w\s]', '', text)
    # Convert text to lowercase
    text = text.lower()
    return text

# Apply the cleaning function to relevant columns
cleaned_dataset.loc[:, 'title'] = cleaned_dataset['title'].apply(clean_text)
cleaned_dataset.loc[:, 'abstract'] = cleaned_dataset['abstract'].apply(clean_text)

# Display the first few rows of the cleaned dataset
cleaned_dataset.head()

,cord_uid,title,abstract
17948,ak20jg32,development and internal validation of a novel...,objective the aim of our study was to identify...
922293,6unac0pg,single domain shark vnar antibodies neutralize...,single domain shark vnar antibodies can offer ...
455876,fcc1cohr,pointofcare ultrasound during the covid19 pand...,purpose the coronavirus disease2019 covid19 le...
775710,73xyjolv,joint predictive value of ctni and ntprobnp on...,background and objectives the pandemic of coro...
115080,k82ifpki,glass hybrid glass ionomer versus composite fo...,objective this study compared survival restora...


# **3. Embedding Preparation**

In [9]:
texts = cleaned_dataset['abstract'].tolist()  # Use the abstract column

# Initialize Tokenizer and Fit on Text
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")  # Use 10,000 most frequent words
tokenizer.fit_on_texts(texts)

# Convert Texts to Sequences
sequences = tokenizer.texts_to_sequences(texts)

# Pad Sequences to Ensure Equal Length
padded_sequences = pad_sequences(sequences, maxlen=100, padding='post', truncating='post')

# Display Example Output
print("Vocabulary Size:", len(tokenizer.word_index))
print("Example Sequence:", sequences[0])
print("Padded Sequence:", padded_sequences[0])

# Convert to TensorFlow tensor for further processing
padded_sequences_tensor = tf.convert_to_tensor(padded_sequences)

Vocabulary Size: 4144
Example Sequence: [206, 2, 321, 3, 66, 24, 10, 5, 263, 108, 4, 1812, 393, 8, 48, 1177, 5, 627, 2, 48, 491, 3, 69, 9, 849, 24, 322, 14, 10, 7, 264, 24, 232, 99, 1178, 2, 1179, 323, 11, 850, 8, 1180, 1813, 1814, 17, 23, 628, 69, 1815, 4, 1816, 1817, 11, 1818, 21, 2, 1819, 3, 265, 233, 160, 394, 4, 629, 8, 86, 161, 92, 21, 851, 4, 38, 265, 79, 4, 629, 852, 630, 266, 49, 10, 492, 631, 493, 20, 1820, 395, 632, 633, 20, 2, 324, 853, 1821, 4, 854, 1822, 1181, 266, 122, 1182, 325, 10, 326, 35, 2, 1823, 267, 8, 1824, 1825, 31, 7, 73, 3, 1183, 12, 8, 327, 1826, 634, 1184, 123, 1185, 855, 4, 494, 1827, 11, 635, 1828, 1829, 1830, 1831, 1832, 148, 1833, 1834, 1835, 4, 1836, 11, 633, 20, 2, 1181, 266, 122, 2, 396, 87, 2, 636, 1837, 19, 2, 630, 122, 10, 1838, 4, 10, 207, 17, 1839, 637, 1840, 325, 109, 22, 234, 4, 856, 7, 638, 495, 122, 5, 857, 2, 1841, 495, 3, 1180, 69, 9, 1184, 4, 397, 7, 1842, 328, 5, 69, 9, 12, 8, 849]
Padded Sequence: [ 206    2  321    3   66   24   10    5

I0000 00:00:1769725188.259322      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


# **4. Output Readiness Check**

In [10]:
# Verify Shape Consistency
print(f"Shape of padded sequences: {padded_sequences.shape}")
assert padded_sequences.ndim == 2, "Data should be 2-dimensional (batch_size, sequence_length)."

# Check for Numerical Format (No Strings or Invalid Data)
print(f"Data type of sequences: {padded_sequences.dtype}")
assert np.issubdtype(padded_sequences.dtype, np.integer), "Data should be integer-encoded."

# Convert to TensorFlow Tensors (if not already)
padded_sequences_tensor = tf.convert_to_tensor(padded_sequences, dtype=tf.int32)

# Verify TensorFlow Data Compatibility
print(f"Tensor shape: {padded_sequences_tensor.shape}")
print(f"Tensor dtype: {padded_sequences_tensor.dtype}")
assert padded_sequences_tensor.dtype == tf.int32, "Tensor dtype should be tf.int32."

# Prepare Data for Batching
batch_size = 32  # Define batch size
dataset = tf.data.Dataset.from_tensor_slices(padded_sequences_tensor).batch(batch_size)

# Check batched data
for batch in dataset.take(1):  # Inspect the first batch
    print(f"Batch shape: {batch.shape}")

Shape of padded sequences: (72, 100)
Data type of sequences: int32
Tensor shape: (72, 100)
Tensor dtype: <dtype: 'int32'>
Batch shape: (32, 100)


# **5. Preparing Data for Sequence-to-Sequence Model**

In [11]:
# Prepare input (titles as keywords) and target (abstracts as papers) sequences
input_texts = cleaned_dataset['title'].tolist()
target_texts = cleaned_dataset['abstract'].tolist()

# Add start and end tokens to targets for generation
target_texts = ['<start> ' + text + ' <end>' for text in target_texts]

# Initialize and fit tokenizer on all texts
all_texts = input_texts + target_texts
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(all_texts)

# Convert texts to sequences
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

# Define max lengths
max_input_len = 50  # For titles
max_target_len = 100  # For abstracts

# Pad input sequences
input_padded = pad_sequences(input_sequences, maxlen=max_input_len, padding='post', truncating='post')

# Pad target sequences
target_padded = pad_sequences(target_sequences, maxlen=max_target_len, padding='post', truncating='post')

# Prepare decoder input and target data
decoder_input_data = target_padded[:, :-1]  # Remove last token
decoder_target_data = target_padded[:, 1:]  # Remove first token (shifted)

# Display shapes
print(f"Input shape: {input_padded.shape}")
print(f"Decoder input shape: {decoder_input_data.shape}")
print(f"Decoder target shape: {decoder_target_data.shape}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

Input shape: (72, 50)
Decoder input shape: (72, 99)
Decoder target shape: (72, 99)
Vocabulary size: 4222


# **6. Building the Transformer Model**

In [12]:
# Model parameters
vocab_size = len(tokenizer.word_index) + 1
d_model = 128
num_heads = 8
num_layers = 4
dff = 512
dropout_rate = 0.1

# Positional Encoding
class PositionalEncoding(Layer):
    def __init__(self, position, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(position, d_model)

    def get_angles(self, position, i, d_model):
        angles = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        return position * angles

    def positional_encoding(self, position, d_model):
        angle_rads = self.get_angles(
            np.arange(position)[:, np.newaxis],
            np.arange(d_model)[np.newaxis, :],
            d_model)
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
        pos_encoding = angle_rads[np.newaxis, ...]
        return tf.cast(pos_encoding, dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

# Multi-Head Attention
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads
        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask):
        batch_size = tf.shape(q)[0]
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        scaled_attention, attention_weights = self.scaled_dot_product_attention(q, k, v, mask)
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        output = self.dense(concat_attention)
        return output, attention_weights

    def scaled_dot_product_attention(self, q, k, v, mask):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        return output, attention_weights

# Feed Forward Network
def point_wise_feed_forward_network(d_model, dff):
    return tf.keras.Sequential([
        Dense(dff, activation='relu'),
        Dense(d_model)
    ])

# Encoder Layer
class EncoderLayer(Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, x, training=False, mask=None):
        attn_output, _ = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

# Encoder
class Encoder(Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, maximum_position_encoding, rate=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = Embedding(input_vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(maximum_position_encoding, d_model)
        self.enc_layers = [EncoderLayer(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.dropout = Dropout(rate)

    def call(self, x, training=False, mask=None):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)
        for i in range(self.num_layers):
            x = self.enc_layers[i](x, training=training, mask=mask)
        return x

# Decoder Layer
class DecoderLayer(Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = point_wise_feed_forward_network(d_model, dff)
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.dropout3 = Dropout(rate)

    def call(self, x, enc_output, training=False, look_ahead_mask=None, padding_mask=None):
        attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layernorm1(attn1 + x)
        attn2, attn_weights_block2 = self.mha2(enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layernorm2(attn2 + out1)
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(ffn_output + out2)
        return out3, attn_weights_block1, attn_weights_block2

# Decoder
class Decoder(Layer):
    def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size, maximum_position_encoding, rate=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.embedding = Embedding(target_vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(maximum_position_encoding, d_model)
        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.dropout = Dropout(rate)

    def call(self, x, enc_output, training=False, look_ahead_mask=None, padding_mask=None):
        seq_len = tf.shape(x)[1]
        attention_weights = {}
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)
        for i in range(self.num_layers):
            x, block1, block2 = self.dec_layers[i](x, enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=padding_mask)
            attention_weights[f'decoder_layer{i+1}_block1'] = block1
            attention_weights[f'decoder_layer{i+1}_block2'] = block2
        return x, attention_weights

# Transformer Model
class Transformer(Model):
    def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size, target_vocab_size, pe_input, pe_target, rate=0.1):
        super(Transformer, self).__init__()
        self.encoder = Encoder(num_layers, d_model, num_heads, dff, input_vocab_size, pe_input, rate)
        self.decoder = Decoder(num_layers, d_model, num_heads, dff, target_vocab_size, pe_target, rate)
        self.final_layer = Dense(target_vocab_size)

    def call(self, inputs, training=None):
        inp, tar = inputs
        enc_padding_mask, look_ahead_mask, dec_padding_mask = self.create_masks(inp, tar)
        enc_output = self.encoder(inp, training=training, mask=enc_padding_mask)
        dec_output, attention_weights = self.decoder(tar, enc_output, training=training, look_ahead_mask=look_ahead_mask, padding_mask=dec_padding_mask)
        final_output = self.final_layer(dec_output)
        return final_output, attention_weights

    def create_masks(self, inp, tar):
        enc_padding_mask = self.create_padding_mask(inp)
        dec_padding_mask = self.create_padding_mask(inp)
        look_ahead_mask = self.create_look_ahead_mask(tf.shape(tar)[1])
        look_ahead_mask = look_ahead_mask[tf.newaxis, tf.newaxis, :, :]  # Expand to (1,1,size,size)
        dec_target_padding_mask = self.create_padding_mask(tar)
        combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)
        return enc_padding_mask, combined_mask, dec_padding_mask

    def create_padding_mask(self, seq):
        seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
        return seq[:, tf.newaxis, tf.newaxis, :]

    def create_look_ahead_mask(self, size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask

# Instantiate the model
transformer = Transformer(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    input_vocab_size=vocab_size,
    target_vocab_size=vocab_size,
    pe_input=max_input_len,
    pe_target=max_target_len,
    rate=dropout_rate)

# Build the model with correct input shapes
transformer.build(input_shape=[(None, max_input_len), (None, max_target_len-2)])

# Summary
transformer.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'transformer', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Encoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Decoder)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# **7. Training the Model**

In [13]:
# Learning rate schedule
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()
        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        arg1 = tf.math.rsqrt(step)
        arg2 = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

# Optimizer
learning_rate = CustomSchedule(d_model)
optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

# Loss function
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

# Metrics
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

# Create dataset
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices((input_padded, decoder_input_data, decoder_target_data))
dataset = dataset.shuffle(buffer_size=len(input_padded)).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)

# Training step
@tf.function
def train_step(inp, dec_inp, tar):
    with tf.GradientTape() as tape:
        predictions, _ = transformer([inp, dec_inp], training=True)
        loss = loss_function(tar, predictions)
    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))
    train_loss(loss)
    train_accuracy(tar, predictions)

# Training loop
epochs = 1000  # Adjust as needed
for epoch in range(epochs):
    train_loss.reset_state()
    train_accuracy.reset_state()
    for (batch, (inp, dec_inp, tar)) in enumerate(dataset):
        train_step(inp, dec_inp, tar)
        if batch % 10 == 0:
            print(f'Epoch {epoch + 1} Batch {batch} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')
    print(f'Epoch {epoch + 1} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')

Epoch 1 Batch 0 Loss 8.3953 Accuracy 0.0000
Epoch 1 Loss 8.3971 Accuracy 0.0000
Epoch 2 Batch 0 Loss 8.3883 Accuracy 0.0003
Epoch 2 Loss 8.3990 Accuracy 0.0003
Epoch 3 Batch 0 Loss 8.3935 Accuracy 0.0000
Epoch 3 Loss 8.3987 Accuracy 0.0000
Epoch 4 Batch 0 Loss 8.3930 Accuracy 0.0000
Epoch 4 Loss 8.3853 Accuracy 0.0003
Epoch 5 Batch 0 Loss 8.3844 Accuracy 0.0003
Epoch 5 Loss 8.3797 Accuracy 0.0003
Epoch 6 Batch 0 Loss 8.3725 Accuracy 0.0000
Epoch 6 Loss 8.3755 Accuracy 0.0001
Epoch 7 Batch 0 Loss 8.3688 Accuracy 0.0000
Epoch 7 Loss 8.3661 Accuracy 0.0000
Epoch 8 Batch 0 Loss 8.3582 Accuracy 0.0000
Epoch 8 Loss 8.3566 Accuracy 0.0000
Epoch 9 Batch 0 Loss 8.3503 Accuracy 0.0000
Epoch 9 Loss 8.3390 Accuracy 0.0001
Epoch 10 Batch 0 Loss 8.3324 Accuracy 0.0000
Epoch 10 Loss 8.3304 Accuracy 0.0000
Epoch 11 Batch 0 Loss 8.3180 Accuracy 0.0000
Epoch 11 Loss 8.3163 Accuracy 0.0000
Epoch 12 Batch 0 Loss 8.3135 Accuracy 0.0000
Epoch 12 Loss 8.3017 Accuracy 0.0000
Epoch 13 Batch 0 Loss 8.2936 Accur

# **8. Inference and Generation**

In [14]:
# Function to generate abstract from title (keywords)
def generate_abstract(title, max_length=100):
    # Tokenize and pad the input title
    input_seq = tokenizer.texts_to_sequences([title])
    input_padded = pad_sequences(input_seq, maxlen=max_input_len, padding='post', truncating='post')
    input_tensor = tf.convert_to_tensor(input_padded, dtype=tf.int32)

    # Start with <start> token
    start_token = tokenizer.word_index.get('<start>', 1)
    end_token = tokenizer.word_index.get('<end>', 2)
    decoder_input = tf.expand_dims([start_token], 0)

    output = []

    for i in range(max_length):
        predictions, _ = transformer([input_tensor, decoder_input], training=False)
        predictions = predictions[:, -1:, :]  # Last token
        predicted_id = tf.argmax(predictions, axis=-1).numpy()[0][0]

        if predicted_id == end_token:
            break

        output.append(predicted_id)
        # Reshape predicted_id to [1, 1] to match decoder_input shape
        predicted_id_expanded = tf.cast([[predicted_id]], tf.int32)
        decoder_input = tf.concat([decoder_input, predicted_id_expanded], axis=1)

    # Convert ids to words
    generated_text = tokenizer.sequences_to_texts([output])[0]
    return generated_text

# Example usage
example_title = "covid 19 vaccine development"
generated_abstract = generate_abstract(example_title)
print(f"Title: {example_title}")
print(f"Generated Abstract: {generated_abstract}")

# Test with a sample from the dataset
sample_title = cleaned_dataset['title'].iloc[0]
actual_abstract = cleaned_dataset['abstract'].iloc[0]
generated = generate_abstract(sample_title)
print(f"\nSample Title: {sample_title}")
print(f"Actual Abstract: {actual_abstract}")
print(f"Generated Abstract: {generated}")

Title: covid 19 vaccine development
Generated Abstract: capable function method is ongoing for nations and its regularities there are intervention that vaccine will be available to

Sample Title: development and internal validation of a novel model to identify inflammatory biomarkers of a response to escitalopram in patients with major depressive disorder
Actual Abstract: objective the aim of our study was to identify immune and inflammationrelated factors with clinical utility to predict the clinical efficacy of treatment for depression study design this was a followup study participants who met the entry criteria were administered with escitalopram 510 mgday as an initial treatment selfevaluation and observer valuations were arranged at the end of weeks 0 4 8 and 12 with blood samples collected at baseline and during weeks 2 and 12 multivariable logistic regression analysis was then carried out by incorporating three cytokines selected by the least absolute shrinkage and selection op